# PySCF Chemistry Integration

This notebook demonstrates building molecular qubit Hamiltonians from
PySCF calculations. The adapter handles SCF, integral extraction,
fermion-to-qubit mapping, and Trotter circuit construction.

**Requirements:** `pip install qdk-pythonic[pyscf]`

## 1. Quick Start: One-Call Interface

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_hamiltonian

h = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", basis="sto-3g")
h.print_summary()

## 2. Step-by-Step Pipeline

For more control, run each stage independently.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import run_scf, get_integrals
from qdk_pythonic.domains.common.fermion import from_integrals
from qdk_pythonic.domains.common.mapping import jordan_wigner
from qdk_pythonic.domains.common.evolution import TrotterEvolution

# Stage 1: Hartree-Fock
mf = run_scf("H 0 0 0; H 0 0 0.74", basis="sto-3g")
print(f"SCF energy: {mf.e_tot:.8f} Hartree")

# Stage 2: Extract integrals
h1e, h2e, nuc = get_integrals(mf)
print(f"Spatial orbitals: {h1e.shape[0]}")
print(f"Nuclear repulsion: {nuc:.6f}")

# Stage 3: Build fermionic Hamiltonian
fermion_op = from_integrals(h1e, h2e, nuc)
print(f"Fermionic terms: {len(fermion_op)}")

# Stage 4: Map to qubits (Jordan-Wigner)
pauli_h = jordan_wigner(fermion_op)
pauli_h.print_summary()

# Stage 5: Build Trotter circuit
evolution = TrotterEvolution(pauli_h, time=1.0, steps=5)
circuit = evolution.to_circuit()
print(f"\nCircuit: {circuit.qubit_count()} qubits, "
      f"{circuit.total_gate_count()} gates, depth {circuit.depth()}")

## 3. Comparing Jordan-Wigner vs Bravyi-Kitaev

In [ ]:
h_jw = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", mapping="jordan_wigner")
h_bk = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", mapping="bravyi_kitaev")

s_jw = h_jw.summary()
s_bk = h_bk.summary()

print(f"{'Mapping':<18} {'Terms':>6} {'Max Weight':>12} {'1-norm':>10}")
print("-" * 50)
print(f"{'Jordan-Wigner':<18} {s_jw['n_terms']:>6} {s_jw['max_pauli_weight']:>12} {s_jw['one_norm']:>10.4f}")
print(f"{'Bravyi-Kitaev':<18} {s_bk['n_terms']:>6} {s_bk['max_pauli_weight']:>12} {s_bk['one_norm']:>10.4f}")

## 4. Active Space Selection

Restrict to a subset of orbitals and electrons to reduce qubit count.

In [ ]:
# LiH: full space vs active space
h_full = molecular_hamiltonian("Li 0 0 0; H 0 0 1.6", basis="sto-3g")
h_active = molecular_hamiltonian(
    "Li 0 0 0; H 0 0 1.6", basis="sto-3g",
    n_active_electrons=2, n_active_orbitals=2,
)

print(f"Full space:   {h_full.qubit_count()} qubits, {len(h_full)} terms")
print(f"Active (2e,2o): {h_active.qubit_count()} qubits, {len(h_active)} terms")

## 5. Using the Algorithm Registry

The QDK/Chemistry-style `create()` pattern works for all adapters.

In [ ]:
from qdk_pythonic.registry import create, available

# See all registered algorithms
for algo_type, names in available().items():
    print(f"{algo_type}: {names}")

In [ ]:
# Build H2 Hamiltonian via registry
builder = create("hamiltonian_builder", "pyscf", basis="sto-3g")
h = builder.run(atom="H 0 0 0; H 0 0 0.74")
h.print_summary()

# Build Trotter circuit via registry
circuit = create("time_evolution_builder", "trotter",
                 time=1.0, steps=5).run(h)
print(f"\nCircuit: {circuit.qubit_count()} qubits, "
      f"{circuit.total_gate_count()} gates")

## 6. End-to-End Summary

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_summary

result = molecular_summary("H 0 0 0; H 0 0 0.74", basis="sto-3g")
for key in ["scf_energy", "n_orbitals", "n_electrons",
            "n_fermion_terms", "n_qubits", "total_gates", "depth"]:
    print(f"{key:<18}: {result[key]}")